# SQL Exercises

Practice the basics: **selecting**, **filtering**, **sorting**, **joining**, and a little **grouping**.

### The data

Four tables:

- **Author** — `Id`, `FirstName`, `LastName`
- **Article** — `Id`, `Title`, `Content`, `AuthorId` → *Author.Id*
- **Tag** — `Id`, `Name`
- **ArticleTag** — `ArticleId` → *Article.Id*, `TagId` → *Tag.Id* (links articles to tags)

### How to answer

Write each answer as a SQL string and pass it to `run(...)`, which returns a pandas DataFrame so you can see the result.

In [10]:
import pandas as pd
import sqlite3

In [11]:
# Setup: connect to the database and define a helper.
conn = sqlite3.connect('articles.db')

def run(query):
    """Run a SQL query and return the result as a pandas DataFrame."""
    return pd.read_sql_query(query, conn)

# Example — try it:
run("SELECT * FROM Article")

,Id,Title,Content,AuthorId
0,1,Politics and Language,An article about the relationship between lang...,1
1,2,Life in the Countryside,"A short reflection on manners, class, and rura...",2
2,3,The River Journey,"A story about travel, humor, and life along th...",3
3,4,The Modern Monster,"An essay about science, ambition, and unintend...",4
4,5,Identity and Society,"A thoughtful piece about identity, justice, an...",5
5,6,Fiction and Truth,An article about how fiction can reveal uncomf...,1
6,7,Satire as Criticism,A short article on using humor to expose hypoc...,3
7,8,The Shape of Fear,An article exploring fear in gothic literature.,4


## 1. Every author

Return **all rows and all columns** from the `Author` table.

In [12]:
query = "SELECT * FROM Author"
run(query)

,Id,FirstName,LastName
0,1,George,Orwell
1,2,Jane,Austen
2,3,Mark,Twain
3,4,Mary,Shelley
4,5,James,Baldwin


## 2. Article titles only

Return just the `Title` column for every article.

In [13]:
query = """
SELECT Title FROM Article;
"""
run(query)

,Title
0,Politics and Language
1,Life in the Countryside
2,The River Journey
3,The Modern Monster
4,Identity and Society
5,Fiction and Truth
6,Satire as Criticism
7,The Shape of Fear


## 3. Filter by author

Return the **titles** of articles whose `AuthorId` is `1`.

In [14]:
query = """
SELECT Title FROM Article
WHERE AuthorId = 1;
"""
run(query)

,Title
0,Politics and Language
1,Fiction and Truth


## 4. Sorted tags

Return every tag's `Name`, sorted alphabetically (A → Z).

In [15]:
query = """
SELECT DISTINCT Name FROM Tag ORDER BY Name ASC;
"""
run(query)

,Name
0,Gothic
1,Literature
2,Politics
3,Satire
4,Science
5,Society
6,Travel


## 5. Sort articles by title

Return all articles, sorted by `Title` (A → Z).

In [17]:
query = """
select * from Article order by Title desc
"""
run(query)

,Id,Title,Content,AuthorId
0,8,The Shape of Fear,An article exploring fear in gothic literature.,4
1,3,The River Journey,"A story about travel, humor, and life along th...",3
2,4,The Modern Monster,"An essay about science, ambition, and unintend...",4
3,7,Satire as Criticism,A short article on using humor to expose hypoc...,3
4,1,Politics and Language,An article about the relationship between lang...,1
5,2,Life in the Countryside,"A short reflection on manners, class, and rura...",2
6,5,Identity and Society,"A thoughtful piece about identity, justice, an...",5
7,6,Fiction and Truth,An article about how fiction can reveal uncomf...,1


## 6. Join: article + author

For every article, return its `Title` along with the author's `FirstName` and `LastName`.

*Hint: `JOIN` the `Article` and `Author` tables where `Article.AuthorId = Author.Id`.*

In [25]:
query = """
select Article.Title, Author.FirstName, Author.LastName 
from Article 
inner join Author on Article.AuthorId = Author.Id;
"""
run(query)

,Title,FirstName,LastName
0,Politics and Language,George,Orwell
1,Life in the Countryside,Jane,Austen
2,The River Journey,Mark,Twain
3,The Modern Monster,Mary,Shelley
4,Identity and Society,James,Baldwin
5,Fiction and Truth,George,Orwell
6,Satire as Criticism,Mark,Twain
7,The Shape of Fear,Mary,Shelley


## 7. Join + filter: articles by Twain

Return the **titles** of every article written by the author whose `LastName` is `'Twain'`.

In [ ]:
query = """
select Article.Title, Author.FirstName, Author.LastName 
from Article 
inner join Author on Article.AuthorId = Author.Id
where Author.LastName='Twain';
"""
run(query)

,Title,FirstName,LastName
0,The River Journey,Mark,Twain
1,Satire as Criticism,Mark,Twain


## 8. Group: articles per author

For each author, return their name and **how many articles** they wrote.

*Hint: `JOIN` `Author` to `Article`, then `GROUP BY` the author and use `COUNT(*)`.*

In [55]:
query = """
select author.lastname as LName, 
count(*)
from author 
join article on author.id=article.authorid 
group by author.id

"""
run(query)

,LName,count(*)
0,Orwell,2
1,Austen,1
2,Twain,2
3,Shelley,2
4,Baldwin,1


## 9. Many-to-many join: article + tags

Return each article's `Title` paired with each `Name` of its tags. An article with two tags should appear twice.

*Hint: this needs two joins — `Article` → `ArticleTag` → `Tag`.*

In [46]:
query = """
select * from article
join articletag on article.id=articletag.articleid
join tag on tag.id=articletag.tagid
"""
run(query)

,Id,Title,Content,AuthorId,ArticleId,TagId,Id,Name
0,1,Politics and Language,An article about the relationship between lang...,1,1,1,1,Politics
1,1,Politics and Language,An article about the relationship between lang...,1,1,4,4,Society
2,2,Life in the Countryside,"A short reflection on manners, class, and rura...",2,2,2,2,Literature
3,2,Life in the Countryside,"A short reflection on manners, class, and rura...",2,2,4,4,Society
4,3,The River Journey,"A story about travel, humor, and life along th...",3,3,3,3,Satire
5,3,The River Journey,"A story about travel, humor, and life along th...",3,3,7,7,Travel
6,4,The Modern Monster,"An essay about science, ambition, and unintend...",4,4,5,5,Science
7,4,The Modern Monster,"An essay about science, ambition, and unintend...",4,4,6,6,Gothic
8,5,Identity and Society,"A thoughtful piece about identity, justice, an...",5,5,4,4,Society
9,5,Identity and Society,"A thoughtful piece about identity, justice, an...",5,5,1,1,Politics


## 10. Group + having: popular tags

Find tags used on **more than one** article. Return the tag `Name` and the number of articles using it, with the most-used tag first.

*Hint: `GROUP BY` the tag, filter groups with `HAVING COUNT(*) > 1`, and `ORDER BY` the count descending.*

In [66]:
query = """
select *, count(*) from article
join articletag on article.id=articletag.articleid
join tag on tag.id=articletag.tagid
group by tag.name
having count(*)>1
order by count(*) desc
"""
run(query)

,Id,Title,Content,AuthorId,ArticleId,TagId,Id,Name,count(*)
0,1,Politics and Language,An article about the relationship between lang...,1,1,4,4,Society,4
1,1,Politics and Language,An article about the relationship between lang...,1,1,1,1,Politics,3
2,2,Life in the Countryside,"A short reflection on manners, class, and rura...",2,2,2,2,Literature,3
3,3,The River Journey,"A story about travel, humor, and life along th...",3,3,3,3,Satire,2
4,4,The Modern Monster,"An essay about science, ambition, and unintend...",4,4,6,6,Gothic,2
